In [ ]:
# Databricks Notebook: 1_Data_Ingestion_and_Preprocessing
# In this notebook, we read raw e-commerce data from CSV/JSON sources,
# perform data cleaning (e.g., handling nulls, type conversions) and save the cleaned data as Delta tables.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, regexp_replace

# Initialize Spark session 
spark = SparkSession.builder.appName("EcommerceDataIngestion").getOrCreate()

# Define data paths
raw_data_path = "dbfs:/mnt/data/raw/ecommerce_sales.csv"  
delta_table_path = "dbfs:/mnt/data/delta/ecommerce_sales_cleaned"

# Read raw data
df_raw = spark.read.option("header", "true").csv(raw_data_path)
display(df_raw)

In [ ]:
# Data cleaning: Convert date columns and remove unwanted characters from price fields
df_cleaned = (
    df_raw
    .withColumn("sale_date", to_date(col("sale_date"), "MM/dd/yyyy"))
    .withColumn("price", regexp_replace(col("price"), "[$,]", "").cast("float"))
    .dropna(subset=["sale_date", "price", "product_id"])
)

# add extra transformations (e.g., categorization, computed columns)
df_cleaned = df_cleaned.withColumn("year", col("sale_date").substr(1,4).cast("int"))

In [ ]:
# Write the cleaned data to a Delta table for further analysis
df_cleaned.write.mode("overwrite").format("delta").save(delta_table_path)
spark.sql("DROP TABLE IF EXISTS ecommerce_sales_cleaned")
spark.sql(f"CREATE TABLE ecommerce_sales_cleaned USING DELTA LOCATION '{delta_table_path}'")

display(df_cleaned)